# Coastal Flood Step 30: Mangrove Priority Ranking (Weighted Area-Distance)

This notebook prioritizes mangrove patches for avoided-damage benefits using the **minimum + maximum scenario weighted signed area-distance attribution outputs**.

Design choices:
- Use **positive avoided EAD** for the priority score to stay comparable with the older protection-shed ranking workflow.
- Keep **negative / increased-damage attribution** separate for diagnostics and net-effect maps.
- Retain weighted-method metadata such as **buffer vs fallback analysis-unit shares**.

Outputs:
- Full ranked table
- No-regret shortlist
- Tiered priority map
- Net-effect maps


In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.patches import Patch

pd.set_option("display.max_columns", 200)

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
robyn_libraries_path = (base_path / "robyns_libraries").resolve()
if str(robyn_libraries_path) not in sys.path:
    sys.path.append(str(robyn_libraries_path))
import Robyn_paper_2_defs

In [ ]:
# -----------------------------
# Parameters
# -----------------------------
TOP_N = 25
TOP_QUARTILE_SHARE = 0.25
TIER1_SHARE = 0.20
TIER2_SHARE = 0.30

# Priority score weights
W_TOTAL = 0.55
W_EFF = 0.30
W_ROBUST = 0.15

mangrove_attribution_buffer_m = 5000
weighted_root_dir = base_path / "dphil_paper_3/results_coastal_scenario_comparison/weighted_area_distance_signed"
out_dir = weighted_root_dir / "mangrove_priority_ranking"
out_dir.mkdir(parents=True, exist_ok=True)

min_damage_dir = base_path / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario/damage_estimates"
max_damage_dir = base_path / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario/damage_estimates"

min_gpkg = min_damage_dir / "mangrove_attribution_area_distance_all_sectors_signed" / f"mangrove_attribution_total_all_sectors_signed_area_distance_{mangrove_attribution_buffer_m}m_nn_fallback.gpkg"
max_gpkg = max_damage_dir / "mangrove_attribution_area_distance_all_sectors_signed" / f"mangrove_attribution_total_all_sectors_signed_area_distance_{mangrove_attribution_buffer_m}m_nn_fallback.gpkg"

boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"

print("Input minimum:", min_gpkg)
print("Input maximum:", max_gpkg)
print("Output folder:", out_dir)

In [ ]:
# Load weighted signed patch outputs
if not min_gpkg.exists():
    raise FileNotFoundError(f"Missing file: {min_gpkg}")
if not max_gpkg.exists():
    raise FileNotFoundError(f"Missing file: {max_gpkg}")
if not boundary_path.exists():
    raise FileNotFoundError(f"Missing file: {boundary_path}")

min_gdf = gpd.read_file(min_gpkg)
max_gdf = gpd.read_file(max_gpkg)
boundary = gpd.read_file(boundary_path)

for gdf in [min_gdf, max_gdf, boundary]:
    if str(gdf.crs).upper() != "EPSG:3448":
        gdf.to_crs("EPSG:3448", inplace=True)

numeric_columns = [
    "Mangrove_Area_m2",
    "Mangrove_Area_ha",
    "Net_Avoided_EAD_USD_attributed",
    "Positive_Avoided_EAD_USD_attributed",
    "Negative_Avoided_EAD_USD_attributed",
    "Absolute_Avoided_EAD_USD_attributed",
    "Asset_Count",
    "Analysis_Unit_Count",
    "Mean_Distance_m",
    "Negative_Avoided_EAD_USD_attributed_abs",
    "Positive_Asset_Count",
    "Negative_Asset_Count",
    "Buffer_Analysis_Unit_Count",
    "Fallback_Analysis_Unit_Count",
]

for patch_gdf in [min_gdf, max_gdf]:
    patch_gdf["Mangrove_ID"] = pd.to_numeric(patch_gdf["Mangrove_ID"], errors="raise").astype(int)
    for column_name in numeric_columns:
        if column_name in patch_gdf.columns:
            patch_gdf[column_name] = pd.to_numeric(patch_gdf[column_name], errors="coerce").fillna(0.0)

min_priority = min_gdf[[
    "Mangrove_ID",
    "Parish",
    "TYPE",
    "Mangrove_Area_ha",
    "Net_Avoided_EAD_USD_attributed",
    "Positive_Avoided_EAD_USD_attributed",
    "Negative_Avoided_EAD_USD_attributed_abs",
    "Asset_Count",
    "Analysis_Unit_Count",
    "Mean_Distance_m",
    "Positive_Asset_Count",
    "Negative_Asset_Count",
    "Buffer_Analysis_Unit_Count",
    "Fallback_Analysis_Unit_Count",
    "geometry",
]].rename(columns={
    "Mangrove_Area_ha": "Mangrove_Area_ha_min",
    "Net_Avoided_EAD_USD_attributed": "net_usd_min",
    "Positive_Avoided_EAD_USD_attributed": "avoided_usd_min",
    "Negative_Avoided_EAD_USD_attributed_abs": "increase_usd_min",
    "Asset_Count": "asset_count_min",
    "Analysis_Unit_Count": "analysis_unit_count_min",
    "Mean_Distance_m": "mean_distance_m_min",
    "Positive_Asset_Count": "positive_asset_count_min",
    "Negative_Asset_Count": "negative_asset_count_min",
    "Buffer_Analysis_Unit_Count": "buffer_unit_count_min",
    "Fallback_Analysis_Unit_Count": "fallback_unit_count_min",
})

max_priority = max_gdf[[
    "Mangrove_ID",
    "Mangrove_Area_ha",
    "Net_Avoided_EAD_USD_attributed",
    "Positive_Avoided_EAD_USD_attributed",
    "Negative_Avoided_EAD_USD_attributed_abs",
    "Asset_Count",
    "Analysis_Unit_Count",
    "Mean_Distance_m",
    "Positive_Asset_Count",
    "Negative_Asset_Count",
    "Buffer_Analysis_Unit_Count",
    "Fallback_Analysis_Unit_Count",
]].rename(columns={
    "Mangrove_Area_ha": "Mangrove_Area_ha_max",
    "Net_Avoided_EAD_USD_attributed": "net_usd_max",
    "Positive_Avoided_EAD_USD_attributed": "avoided_usd_max",
    "Negative_Avoided_EAD_USD_attributed_abs": "increase_usd_max",
    "Asset_Count": "asset_count_max",
    "Analysis_Unit_Count": "analysis_unit_count_max",
    "Mean_Distance_m": "mean_distance_m_max",
    "Positive_Asset_Count": "positive_asset_count_max",
    "Negative_Asset_Count": "negative_asset_count_max",
    "Buffer_Analysis_Unit_Count": "buffer_unit_count_max",
    "Fallback_Analysis_Unit_Count": "fallback_unit_count_max",
})

priority = min_priority.merge(max_priority, on="Mangrove_ID", how="outer")
priority["Parish"] = priority["Parish"].fillna("Unassigned")
priority["TYPE"] = priority["TYPE"].fillna("Unknown")
priority["Mangrove_Area_ha"] = priority["Mangrove_Area_ha_min"].fillna(priority["Mangrove_Area_ha_max"]).fillna(0.0)

fill_zero_columns = [
    "avoided_usd_min",
    "avoided_usd_max",
    "increase_usd_min",
    "increase_usd_max",
    "net_usd_min",
    "net_usd_max",
    "asset_count_min",
    "asset_count_max",
    "analysis_unit_count_min",
    "analysis_unit_count_max",
    "mean_distance_m_min",
    "mean_distance_m_max",
    "positive_asset_count_min",
    "positive_asset_count_max",
    "negative_asset_count_min",
    "negative_asset_count_max",
    "buffer_unit_count_min",
    "buffer_unit_count_max",
    "fallback_unit_count_min",
    "fallback_unit_count_max",
]
for column_name in fill_zero_columns:
    priority[column_name] = priority[column_name].fillna(0.0)

priority = gpd.GeoDataFrame(priority, geometry="geometry", crs="EPSG:3448")
priority["Mangrove_Label"] = priority["Mangrove_ID"].astype(str) + " | " + priority["Parish"].astype(str)

print("Patches loaded:", len(priority))
display(priority[["Mangrove_ID", "Parish", "avoided_usd_min", "avoided_usd_max", "increase_usd_min", "increase_usd_max"]].head())

In [ ]:
# Metrics
priority["area_ha"] = priority.geometry.area / 10000.0
priority["area_ha"] = priority["area_ha"].where(priority["area_ha"] > 0, priority["Mangrove_Area_ha"])

priority["avoided_usd_mean"] = (priority["avoided_usd_min"] + priority["avoided_usd_max"]) / 2.0
priority["avoided_usd_range"] = priority["avoided_usd_max"] - priority["avoided_usd_min"]
priority["increase_usd_mean"] = (priority["increase_usd_min"] + priority["increase_usd_max"]) / 2.0
priority["net_usd_mean"] = (priority["net_usd_min"] + priority["net_usd_max"]) / 2.0

priority["usd_per_ha_min"] = np.where(priority["area_ha"] > 0, priority["avoided_usd_min"] / priority["area_ha"], np.nan)
priority["usd_per_ha_max"] = np.where(priority["area_ha"] > 0, priority["avoided_usd_max"] / priority["area_ha"], np.nan)
priority["usd_per_ha_mean"] = np.where(priority["area_ha"] > 0, priority["avoided_usd_mean"] / priority["area_ha"], np.nan)
priority["net_usd_per_ha_mean"] = np.where(priority["area_ha"] > 0, priority["net_usd_mean"] / priority["area_ha"], np.nan)

priority["has_increase_min"] = priority["increase_usd_min"] > 0
priority["has_increase_max"] = priority["increase_usd_max"] > 0
priority["has_increase_any"] = priority["has_increase_min"] | priority["has_increase_max"]
priority["has_net_negative_min"] = priority["net_usd_min"] < 0
priority["has_net_negative_max"] = priority["net_usd_max"] < 0
priority["has_net_negative_any"] = priority["has_net_negative_min"] | priority["has_net_negative_max"]

priority["buffer_share_pct_min"] = np.where(
    priority["analysis_unit_count_min"] > 0,
    100.0 * priority["buffer_unit_count_min"] / priority["analysis_unit_count_min"],
    np.nan,
)
priority["fallback_share_pct_min"] = np.where(
    priority["analysis_unit_count_min"] > 0,
    100.0 * priority["fallback_unit_count_min"] / priority["analysis_unit_count_min"],
    np.nan,
)
priority["buffer_share_pct_max"] = np.where(
    priority["analysis_unit_count_max"] > 0,
    100.0 * priority["buffer_unit_count_max"] / priority["analysis_unit_count_max"],
    np.nan,
)
priority["fallback_share_pct_max"] = np.where(
    priority["analysis_unit_count_max"] > 0,
    100.0 * priority["fallback_unit_count_max"] / priority["analysis_unit_count_max"],
    np.nan,
)

priority["mean_distance_m_mean"] = (priority["mean_distance_m_min"] + priority["mean_distance_m_max"]) / 2.0

priority["rank_min"] = priority["avoided_usd_min"].rank(method="dense", ascending=False)
priority["rank_max"] = priority["avoided_usd_max"].rank(method="dense", ascending=False)
priority["rank_mean"] = priority["avoided_usd_mean"].rank(method="dense", ascending=False)
priority["rank_eff"] = priority["usd_per_ha_mean"].rank(method="dense", ascending=False)
priority["rank_volatility"] = (priority["rank_min"] - priority["rank_max"]).abs()

priority_count = len(priority)
if priority_count <= 1:
    priority["robustness_norm"] = 1.0
else:
    priority["robustness_norm"] = 1.0 - (priority["rank_volatility"] / (priority_count - 1))

def minmax(series):
    series_min = float(series.min())
    series_max = float(series.max())
    if np.isclose(series_max, series_min):
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series_min) / (series_max - series_min)

priority["total_norm"] = minmax(priority["avoided_usd_mean"])
priority["eff_norm"] = minmax(priority["usd_per_ha_mean"].fillna(0.0))
priority["priority_score"] = (
    W_TOTAL * priority["total_norm"] +
    W_EFF * priority["eff_norm"] +
    W_ROBUST * priority["robustness_norm"]
)
priority["priority_rank"] = priority["priority_score"].rank(method="dense", ascending=False)

In [ ]:
# No-regret and tier flags
priority_count = len(priority)
rank_cut = max(1, int(np.ceil(TOP_QUARTILE_SHARE * priority_count)))

def tier_from_rank(rank_value):
    tier_1_cut = max(1, int(np.ceil(TIER1_SHARE * priority_count)))
    tier_2_cut = max(tier_1_cut + 1, int(np.ceil((TIER1_SHARE + TIER2_SHARE) * priority_count)))
    if rank_value <= tier_1_cut:
        return "Tier 1 (Highest)"
    if rank_value <= tier_2_cut:
        return "Tier 2 (Medium)"
    return "Tier 3 (Lower)"

priority["no_regret"] = (
    (priority["rank_min"] <= rank_cut) &
    (priority["rank_max"] <= rank_cut) &
    (priority["avoided_usd_min"] > 0) &
    (priority["avoided_usd_max"] > 0)
)
priority["no_regret_net_positive"] = (
    priority["no_regret"] &
    (priority["net_usd_min"] > 0) &
    (priority["net_usd_max"] > 0)
)

priority["priority_tier"] = priority["priority_rank"].apply(tier_from_rank)
priority = priority.sort_values(["priority_rank", "rank_mean", "Mangrove_ID"]).reset_index(drop=True)

cols_show = [
    "Mangrove_ID",
    "Parish",
    "priority_rank",
    "priority_tier",
    "no_regret",
    "no_regret_net_positive",
    "avoided_usd_min",
    "avoided_usd_max",
    "avoided_usd_mean",
    "increase_usd_min",
    "increase_usd_max",
    "net_usd_min",
    "net_usd_max",
    "usd_per_ha_mean",
    "buffer_share_pct_min",
    "fallback_share_pct_min",
    "buffer_share_pct_max",
    "fallback_share_pct_max",
    "rank_min",
    "rank_max",
    "rank_mean",
    "rank_volatility",
    "priority_score",
]

display(priority[cols_show].head(TOP_N))
print("Patches with any increased damage:", int(priority["has_increase_any"].sum()))
print("Patches with any net negative effect:", int(priority["has_net_negative_any"].sum()))

In [ ]:
# Save outputs
full_csv = out_dir / "mangrove_priority_full_table_weighted_area_distance.csv"
short_csv = out_dir / f"mangrove_priority_top_{TOP_N}_weighted_area_distance.csv"
noregret_csv = out_dir / "mangrove_priority_no_regret_weighted_area_distance.csv"
increase_csv = out_dir / "mangrove_priority_increased_damage_flags_weighted_area_distance.csv"
summary_csv = out_dir / "mangrove_priority_summary_weighted_area_distance.csv"
map_gpkg = out_dir / "mangrove_priority_map_weighted_area_distance.gpkg"

priority.to_csv(full_csv, index=False)
priority.head(TOP_N).to_csv(short_csv, index=False)
priority.loc[priority["no_regret"]].to_csv(noregret_csv, index=False)
priority[[
    "Mangrove_ID",
    "increase_usd_min",
    "increase_usd_max",
    "has_increase_min",
    "has_increase_max",
    "has_increase_any",
    "has_net_negative_min",
    "has_net_negative_max",
    "has_net_negative_any",
]].to_csv(increase_csv, index=False)
priority.to_file(map_gpkg, driver="GPKG")

summary_df = pd.DataFrame([
    {
        "total_patches": len(priority),
        "total_positive_avoided_usd_min": float(priority["avoided_usd_min"].sum()),
        "total_positive_avoided_usd_max": float(priority["avoided_usd_max"].sum()),
        "total_net_usd_min": float(priority["net_usd_min"].sum()),
        "total_net_usd_max": float(priority["net_usd_max"].sum()),
        "no_regret_count": int(priority["no_regret"].sum()),
        "no_regret_net_positive_count": int(priority["no_regret_net_positive"].sum()),
        "tier1_count": int((priority["priority_tier"] == "Tier 1 (Highest)").sum()),
        "tier2_count": int((priority["priority_tier"] == "Tier 2 (Medium)").sum()),
        "tier3_count": int((priority["priority_tier"] == "Tier 3 (Lower)").sum()),
        "patches_with_increased_damage": int(priority["has_increase_any"].sum()),
        "patches_with_net_negative_effect": int(priority["has_net_negative_any"].sum()),
    }
])
summary_df.to_csv(summary_csv, index=False)

print("Saved:")
print(" -", full_csv)
print(" -", short_csv)
print(" -", noregret_csv)
print(" -", increase_csv)
print(" -", summary_csv)
print(" -", map_gpkg)

In [ ]:
# Summary stats
summary_df

In [ ]:
# Plot: top-N bar chart
plot_df = priority.head(TOP_N).copy().sort_values("priority_rank", ascending=False)

tier_colors = {
    "Tier 1 (Highest)": "#2E7D32",
    "Tier 2 (Medium)": "#F9A825",
    "Tier 3 (Lower)": "#B0BEC5",
}
bar_colors = plot_df["priority_tier"].map(tier_colors).fillna("#607D8B")
bar_hatches = np.where(plot_df["has_increase_any"], "////", "")

fig, axis = plt.subplots(figsize=(10, max(6, TOP_N * 0.24)))
bars = axis.barh(plot_df["Mangrove_Label"], plot_df["avoided_usd_mean"], color=bar_colors, edgecolor="black", linewidth=0.2)
for bar, hatch in zip(bars, bar_hatches):
    bar.set_hatch(hatch)

axis.set_title(f"Top {TOP_N} Mangrove Patches by Mean Positive Avoided EAD (Weighted)")
axis.set_xlabel("Mean positive avoided EAD (USD)")
axis.set_ylabel("Mangrove patch")
axis.grid(axis="x", alpha=0.25, linestyle="--")

legend_handles = [
    Patch(facecolor=tier_colors[tier_name], edgecolor="black", label=tier_name)
    for tier_name in ["Tier 1 (Highest)", "Tier 2 (Medium)", "Tier 3 (Lower)"]
]
legend_handles.append(Patch(facecolor="white", edgecolor="black", hatch="////", label="Any increased damage"))
axis.legend(handles=legend_handles, frameon=True, title="Priority layers", loc="lower right")

plt.tight_layout()
bar_png = out_dir / f"mangrove_priority_top_{TOP_N}_bar_weighted_area_distance.png"
fig.savefig(bar_png, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", bar_png)

In [ ]:
# Plot: tiered maps (minimum and maximum increased damage shown separately)
tier_colors = {
    "Tier 1 (Highest)": "#2E7D32",
    "Tier 2 (Medium)": "#F9A825",
    "Tier 3 (Lower)": "#B0BEC5",
}

def style_jamaica_axis(axis, title_text):
    boundary.boundary.plot(ax=axis, color="black", linewidth=0.8)
    Robyn_paper_2_defs.draw_scale_bar(axis, location=(0.88, 0.78), length_km=20, linewidth=0.6, label_offset=0.02, km_offset=0.01)
    Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)
    axis.set_title(title_text)
    axis.set_axis_off()

def plot_tier_map_with_increase(increase_col, increase_label, out_name, title_text):
    fig, axis = plt.subplots(figsize=(8, 10))

    for tier_name, color in tier_colors.items():
        subset = priority.loc[priority["priority_tier"] == tier_name]
        if len(subset) > 0:
            subset.plot(ax=axis, color=color, edgecolor="black", linewidth=0.1)

    increased = priority.loc[priority[increase_col]]
    if len(increased) > 0:
        increased.plot(ax=axis, facecolor="none", edgecolor="#C62828", linewidth=0.9, hatch="////")

    style_jamaica_axis(axis, title_text)
    legend_handles = [
        Patch(facecolor=tier_colors[tier_name], edgecolor="black", label=tier_name)
        for tier_name in ["Tier 1 (Highest)", "Tier 2 (Medium)", "Tier 3 (Lower)"]
    ]
    legend_handles.append(Patch(facecolor="white", edgecolor="#C62828", hatch="////", label=increase_label))
    axis.legend(handles=legend_handles, loc="lower left", frameon=True, title="Map layers")

    plt.tight_layout()
    out_png = out_dir / out_name
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png)

plot_tier_map_with_increase(
    increase_col="has_increase_min",
    increase_label="Increased damage (minimum scenario)",
    out_name="mangrove_priority_tier_map_weighted_area_distance_minimum.png",
    title_text="Mangrove Priority Tiers + Increased-Damage Hotspots (Weighted, Minimum)",
)

plot_tier_map_with_increase(
    increase_col="has_increase_max",
    increase_label="Increased damage (maximum scenario)",
    out_name="mangrove_priority_tier_map_weighted_area_distance_maximum.png",
    title_text="Mangrove Priority Tiers + Increased-Damage Hotspots (Weighted, Maximum)",
)

In [ ]:
# Plot: net effect maps (net positive vs net negative), minimum and maximum separately)
def classify_net(value):
    if value > 0:
        return "Net positive"
    if value < 0:
        return "Net negative"
    return "Neutral (0)"

priority["net_class_min"] = priority["net_usd_min"].apply(classify_net)
priority["net_class_max"] = priority["net_usd_max"].apply(classify_net)

net_colors = {
    "Net positive": "#2E7D32",
    "Net negative": "#C62828",
    "Neutral (0)": "#CFD8DC",
}

def plot_net_map(class_col, out_name, title_text):
    fig, axis = plt.subplots(figsize=(8, 10))

    for class_name in ["Neutral (0)", "Net positive", "Net negative"]:
        subset = priority.loc[priority[class_col] == class_name]
        if len(subset) > 0:
            subset.plot(ax=axis, color=net_colors[class_name], edgecolor="black", linewidth=0.1)

    style_jamaica_axis(axis, title_text)
    handles = [
        Patch(facecolor=net_colors["Net positive"], edgecolor="black", label="Net positive (positive > increase)"),
        Patch(facecolor=net_colors["Net negative"], edgecolor="black", label="Net negative (increase > positive)"),
        Patch(facecolor=net_colors["Neutral (0)"], edgecolor="black", label="Neutral (equal/zero)"),
    ]
    axis.legend(handles=handles, loc="lower left", frameon=True, title="Net effect")

    plt.tight_layout()
    out_png = out_dir / out_name
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png)

plot_net_map(
    class_col="net_class_min",
    out_name="mangrove_priority_net_effect_map_weighted_area_distance_minimum.png",
    title_text="Mangrove Patch Net Flood-Risk Effect (Weighted, Minimum)",
)

plot_net_map(
    class_col="net_class_max",
    out_name="mangrove_priority_net_effect_map_weighted_area_distance_maximum.png",
    title_text="Mangrove Patch Net Flood-Risk Effect (Weighted, Maximum)",
)

print("Minimum map counts:", priority["net_class_min"].value_counts().to_dict())
print("Maximum map counts:", priority["net_class_max"].value_counts().to_dict())